In [26]:
import os
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Plot simulation

In [33]:
num_hosts = 16
nodes = 16 # This is the nodes used for greedy local algorithm, 1 if global
processors_per_host = 36
processors = num_hosts * processors_per_host

groupings = {
    "Round Robin": "test/processor_group/c48_p576/groups_mod16.csv",
    "Greedy Group": "test/processor_group/c48_p576/greedy_groups.csv",
    "Original Group": None
}

test_name_base = f"greedy/c180_p{processors}"

test_names = {}
seeds = {}
pathes = {}
for key, grouping in groupings.items():
    test_name = test_name_base
    seeds[key] = 0
    if nodes > 1:
        test_name += f"_h{nodes}"
        if grouping:
            if grouping.isdigit():
                seed = int(grouping)
                seeds[key] = seed
                test_name += f"_s{seed}"
            else:
                path = Path(grouping)
                pathes[key] = path
                test_name += f"_f{path.stem}"
    test_names[key] = test_name

test_names

{'Round Robin': 'greedy/c180_p576_h16_fgroups_mod16',
 'Greedy Group': 'greedy/c180_p576_h16_fgreedy_groups',
 'Original Group': 'greedy/c180_p576_h16'}

In [34]:
simulation_data = {}
for key, test_name in test_names.items():
    simulation_data_path = f"test/{test_name}/simulation.csv"
    simulation_data[key] = pd.read_csv(simulation_data_path, index_col=0)
    # Exclude the last four columns and select only numeric columns
    simulation_data[key] = simulation_data[key].iloc[:, :-4].select_dtypes(include=[np.number])

# Print the first item in simulation_data dict
first_key = next(iter(simulation_data))
simulation_data[first_key]

,Processor0,Processor1,Processor2,Processor3,Processor4,Processor5,Processor6,Processor7,Processor8,Processor9,...,Processor566,Processor567,Processor568,Processor569,Processor570,Processor571,Processor572,Processor573,Processor574,Processor575
Interval,,,,,,,,,,,,,,,,,,,,,
0,67719.0,64202.0,69095.0,59486.0,61241.0,72507.0,84960.0,82958.0,69345.0,64537.0,...,80351.0,135586.0,66338.0,68251.0,70630.0,62795.0,63195.0,72814.0,78392.0,81774.0
1,48550.0,52938.0,53764.0,53730.0,53851.0,60106.0,62511.0,59443.0,55675.0,53429.0,...,61218.0,70287.0,55567.0,53100.0,55144.0,52931.0,56152.0,57109.0,62258.0,63842.0
2,45395.0,51962.0,52082.0,52541.0,55397.0,57767.0,62378.0,57593.0,53091.0,51295.0,...,60352.0,64448.0,53619.0,52324.0,52349.0,52050.0,55422.0,54494.0,59920.0,56504.0
3,44347.0,50915.0,50657.0,52048.0,55404.0,56308.0,62111.0,56553.0,51615.0,50414.0,...,61468.0,58987.0,51777.0,52141.0,65164.0,52348.0,55120.0,52658.0,58968.0,57081.0
4,44825.0,49281.0,51532.0,51447.0,55463.0,55296.0,60296.0,53346.0,50793.0,50464.0,...,58789.0,61116.0,51518.0,52672.0,61490.0,52493.0,54567.0,54039.0,56317.0,52982.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,65691.0,58189.0,51423.0,49321.0,47422.0,51709.0,63860.0,73616.0,68441.0,63721.0,...,67926.0,78307.0,87020.0,65006.0,53809.0,48052.0,46355.0,50212.0,62825.0,72621.0
500,65790.0,56697.0,51563.0,48718.0,47199.0,55634.0,62652.0,75538.0,62766.0,60619.0,...,64710.0,77653.0,69800.0,63721.0,55518.0,48330.0,46930.0,51427.0,64029.0,72130.0
501,65125.0,55292.0,51612.0,49446.0,47697.0,57648.0,67108.0,71993.0,59146.0,58697.0,...,72437.0,76496.0,82835.0,63435.0,57895.0,46212.0,48117.0,52916.0,63035.0,70283.0


In [29]:
# Plot the simulation data as a heatmap,
def plot_simulation_heatmap(data: pd.DataFrame, heatmap_type: str = "processor", path: Optional[str] = None):
    """
    Plot a heatmap of the simulation data.
    Args:
        data (pd.DataFrame): The data to plot.
        heatmap_type (str): Label for the x-axis (e.g., 'processor', 'node').
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(12, 8))
    cax = plt.imshow(data, aspect="auto", cmap="viridis")
    plt.colorbar(cax, label="Value")
    plt.xlabel(heatmap_type.capitalize())
    plt.ylabel("Interval")
    plt.title(f"{heatmap_type.capitalize()} Data Heatmap")
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/{heatmap_type}_heatmap.png")
        plt.savefig(f"{path}/{heatmap_type}_heatmap.eps")
    plt.show()

In [30]:
# # Plot the simulation data heatmap
# plot_simulation_heatmap(
#     simulation_data, heatmap_type="processor", path=f"test/plots/{test_name}"
# )

Generate random grouping or read from input file

In [31]:
def generate_group_from_seed(
    num_hosts: int, processors_per_host: int, seed: int = 0
 ) -> pd.DataFrame:
    """
    Generate processor groups based on a seed value.
    Args:
        seed (int): Seed for random number generator.
        num_hosts (int): Number of hosts.
        processors_per_host (int): Number of processors per host.
    Returns:
        pd.DataFrame: DataFrame with processor groups.
    """
    processors = num_hosts * processors_per_host
    indices = np.arange(processors)
    if seed:
        np.random.seed(seed)
        np.random.shuffle(indices)
    processor_groups = [
        list(indices[i * processors_per_host : (i + 1) * processors_per_host])
        for i in range(num_hosts)
    ]
    return pd.DataFrame(processor_groups)

node_dfs = {}
for key, value in test_names.items():
    if key in pathes:
        node_dfs[key] = pd.read_csv(pathes[key], header=None)
    elif key in seeds:
        node_dfs[key] = generate_group_from_seed(
            num_hosts, processors_per_host, seeds[key]
        )

# Print the first item in node_dfs dict
first_key = next(iter(node_dfs))
node_dfs[first_key]

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,0,16,32,48,64,80,96,112,128,144,...,416,432,448,464,480,496,512,528,544,560
1,1,17,33,49,65,81,97,113,129,145,...,417,433,449,465,481,497,513,529,545,561
2,2,18,34,50,66,82,98,114,130,146,...,418,434,450,466,482,498,514,530,546,562
3,3,19,35,51,67,83,99,115,131,147,...,419,435,451,467,483,499,515,531,547,563
4,4,20,36,52,68,84,100,116,132,148,...,420,436,452,468,484,500,516,532,548,564
5,5,21,37,53,69,85,101,117,133,149,...,421,437,453,469,485,501,517,533,549,565
6,6,22,38,54,70,86,102,118,134,150,...,422,438,454,470,486,502,518,534,550,566
7,7,23,39,55,71,87,103,119,135,151,...,423,439,455,471,487,503,519,535,551,567
8,8,24,40,56,72,88,104,120,136,152,...,424,440,456,472,488,504,520,536,552,568
9,9,25,41,57,73,89,105,121,137,153,...,425,441,457,473,489,505,521,537,553,569


Plot group's total simulated workload

In [35]:
dynamic_summed_dfs = {}

for key, node_df in node_dfs.items():
    dynamic_node_mapping = {}
    for group_id, row in node_df.iterrows():
        for processor_id in row:
            dynamic_node_mapping[f"Processor{processor_id}"] = group_id

    # Rename columns in workload_df using the dynamic node mapping
    dynamic_grouped_df = simulation_data[key].rename(columns=dynamic_node_mapping)
    # Sum workload by group
    dynamic_summed_dfs[key] = dynamic_grouped_df.T.groupby(level=0).sum().T

# Print the first item in dynamic_summed_dfs dict
first_key = next(iter(dynamic_summed_dfs))
dynamic_summed_dfs[first_key]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
Interval,,,,,,,,,,,,,,,,
0,2870539.0,2841276.0,2874125.0,2742288.0,2801944.0,3160521.0,3471259.0,3345545.0,2881006.0,2881566.0,2877363.0,2765184.0,2829271.0,3124308.0,3341030.0,3193987.0
1,2401724.0,2552919.0,2541614.0,2385296.0,2513986.0,2788492.0,2899254.0,2717490.0,2449576.0,2602336.0,2678233.0,2417345.0,2538708.0,2800282.0,2859908.0,2657581.0
2,2328730.0,2531000.0,2612921.0,2410660.0,2484128.0,2686272.0,2746913.0,2550690.0,2351890.0,2615713.0,2730007.0,2434214.0,2585251.0,2698092.0,2735417.0,2617861.0
3,2302361.0,2598681.0,2686443.0,2419370.0,2465596.0,2591111.0,2650505.0,2530832.0,2262435.0,2657968.0,2774630.0,2505437.0,2573030.0,2524504.0,2643471.0,2527156.0
4,2319377.0,2746430.0,2722350.0,2493859.0,2465473.0,2475236.0,2588067.0,2548728.0,2267525.0,2691128.0,2796358.0,2577006.0,2465419.0,2424119.0,2556576.0,2446380.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,3007049.0,2558417.0,2119573.0,2165380.0,2440619.0,2621254.0,3157846.0,3617113.0,3072679.0,2727205.0,2170708.0,2161578.0,2346243.0,2638719.0,3020926.0,3312057.0
500,2849284.0,2559361.0,2160687.0,2175093.0,2416888.0,2642980.0,3153758.0,3572491.0,2974048.0,2679994.0,2143673.0,2142753.0,2396021.0,2635206.0,3103019.0,3220717.0
501,2730815.0,2591173.0,2183846.0,2187807.0,2433535.0,2711528.0,3156217.0,3368317.0,2850460.0,2628531.0,2157125.0,2177631.0,2413206.0,2622899.0,3180566.0,3121765.0


In [40]:
# Max group workload per interval
max_group_workloads = {}
max_group_workloads_per_processor = {}
for key, dynamic_summed_df in dynamic_summed_dfs.items():
    max_group_workload = dynamic_summed_df.max(axis=1)
    max_group_workloads[key] = max_group_workload
    max_group_workloads_per_processor[key] = max_group_workload / processors_per_host

# Print the first item in max_group_workloads_per_processor dict
first_key = next(iter(max_group_workloads_per_processor))
max_group_workloads_per_processor[first_key]

Interval
0       96423.861111
1       80534.833333
2       76303.138889
3       77073.055556
4       77676.611111
           ...      
499    100475.361111
500     99235.861111
501     93564.361111
502     88777.805556
503     87978.222222
Length: 504, dtype: float64

In [45]:
# Span is the sum of the max group workload across all intervals, divided by the number of processors per host
spans = {}
for key, max_group_workload in max_group_workloads.items():
    spans[key] = max_group_workload.sum()

spans

{'Round Robin': np.float64(1586522679.0),
 'Greedy Group': np.float64(1578230363.0),
 'Original Group': np.float64(1655282179.0)}

In [47]:
# Plot the span as a bar chart
def plot_span_bar_chart(
    spans: dict, path: Optional[str] = None
):
    """
    Plot a bar chart of the span for each test.
    Args:
        spans (dict): Dictionary with test names as keys and spans as values.
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(10, 6))
    plt.bar(spans.keys(), spans.values(), color='skyblue')
    plt.xlabel('Test Name')
    plt.ylabel('Span')
    plt.title('Span for Each Test')
    plt.xticks(rotation=45)
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/span_bar_chart.png")
        plt.savefig(f"{path}/span_bar_chart.eps")

# Plot the span bar chart
plot_span_bar_chart(
    spans, path=f"test/plots/{test_name_base}"
)

Workload

In [ ]:
# Plot the node workload as a bar chart for one interval
def plot_node_workload_bar_chart(data: pd.Series, interval: int, path: Optional[str] = None):
    """
    Plot a bar chart of the node workload for a specific interval.
    Args:
        data (pd.Series): The data to plot.
        interval (int): The interval to plot.
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(12, 6))
    data.plot(kind='bar', label='Node')
    plt.title(f"C180 Node Workload for Interval {interval}")
    plt.xlabel("Node")
    plt.ylabel("Workload")
    # Add a reference line at y = mean workload
    mean_workload = data.mean()
    plt.axhline(mean_workload, color='red', linestyle='--', label='Mean')
    plt.legend()
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/node_workload_interval_{interval}.png")
        plt.savefig(f"{path}/node_workload_interval_{interval}.eps")
    plt.show()

In [14]:
# Plot multiple max node workloads as a line chart for comparison
def plot_max_node_workload(series_map: dict, path: Optional[str] = None):
    """
    Plot multiple maximum node workloads as a line chart for comparison.
    Args:
        series_map (dict): Dictionary mapping label (str) to pd.Series (max workload per node).
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(12, 6))
    for label, data in series_map.items():
        data.plot(kind='line', marker='o', label=label)
    plt.title("Maximum Node Workload Comparison")
    plt.xlabel("Interval")
    plt.ylabel("Max Workload")
    plt.legend()
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/max_node_workload_comparison.png")
        plt.savefig(f"{path}/max_node_workload_comparison.eps")
    plt.show()

In [42]:
# Normalize workload to original group
normalized_max_group_workloads = {}
if "Original Group" in max_group_workloads:
    original_max_group_workload = max_group_workloads["Original Group"]
else:
    raise ValueError("Original Group not found in max_group_workloads")

for key, max_group_workload in max_group_workloads.items():
    normalized_max_group_workloads[key] = max_group_workload / original_max_group_workload

normalized_max_group_workloads["Original Group"]

Interval
0      1.0
1      1.0
2      1.0
3      1.0
4      1.0
      ... 
499    1.0
500    1.0
501    1.0
502    1.0
503    1.0
Length: 504, dtype: float64

In [44]:
plot_max_node_workload(normalized_max_group_workloads, path=f"test/plots/{test_name_base}")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
